# SDH exp_024 — Exact-event EB train-only OOF 검증

LB를 다시 사용하지 않습니다. `test.csv`를 읽지 않고 H0와 Exact-event EB를 동일한 5-fold × 3-seed 계약으로 비교합니다.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'raw' / 'train.csv').exists():
    ROOT = next((p for p in [ROOT, *ROOT.parents] if (p / 'data' / 'raw' / 'train.csv').exists()), None)
if ROOT is None:
    raise RuntimeError('저장소 루트 또는 저장소 내부에서 실행해 주세요.')
EXP_DIR = ROOT / 'experiments' / 'SDH' / 'exp_024_exact_event_eb_reproduction'
RESULT_DIR = EXP_DIR / 'results'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
if str(EXP_DIR) not in sys.path:
    sys.path.insert(0, str(EXP_DIR))
import exact_event_pipeline as pipeline
import oof_validation as exp
print('ROOT:', ROOT)
print('seeds:', exp.SEEDS)


## 1. Train만 로드

이 노트북에는 `test.csv` 로드 코드가 없습니다. 모든 supervised 피처는 outer-fold train 내부에서 cross-fit됩니다.

In [ ]:
train = pd.read_csv(ROOT / 'data' / 'raw' / 'train.csv')
genes = [column for column in train.columns if column not in ('ID', 'SUBCLASS')]
assert list(train.columns) == ['ID', 'SUBCLASS', *genes]
assert train[genes].isna().sum().sum() == 0
print('train:', train.shape, 'genes:', len(genes), 'classes:', train.SUBCLASS.nunique())


## 2. Seed 42

각 fold에서 H0와 Exact-event EB가 동일한 학습/검증 행을 사용합니다. 가장 먼저 방향성을 확인합니다.

In [ ]:
result_42 = exp.run_seed(train, genes, seed=42)
pd.DataFrame(result_42.scores()).T


In [ ]:
result_42.fold_metrics[['fold', 'h0_f1', 'final_exact_f1', 'delta_vs_h0', 'exact_vocabulary_size']]


## 3. Seed 777

In [ ]:
result_777 = exp.run_seed(train, genes, seed=777)
pd.DataFrame(result_777.scores()).T


## 4. Seed 2024

In [ ]:
result_2024 = exp.run_seed(train, genes, seed=2024)
pd.DataFrame(result_2024.scores()).T


## 5. 3-seed 집계 및 채택 판정

평균 개선, 최소 seed 개선, 양수 seed 수, 양수 fold 수를 한 번에 확인합니다.

In [ ]:
results = [result_42, result_777, result_2024]
per_seed, summary, decision = exp.aggregate(results)
display(per_seed)
display(summary)
display(decision)


## 6. 클래스별 개선·하락 분석

`recovered`는 H0가 틀리고 Exact가 맞힌 수, `broken`은 그 반대입니다. 세 seed 합계입니다.

In [ ]:
class_result = exp.class_comparison(results)
display(class_result.head(10))
display(class_result.tail(10))


## 7. 결과 저장

OOF 확률 자체는 저장·커밋하지 않고 경량 metrics, fold, class, audit만 `results/`에 저장합니다.

In [ ]:
report = exp.save_results(results, RESULT_DIR)
report


## 8. 선택 실행 — permutation-label sanity check

추가로 1-seed × 5-fold가 돌아 오래 걸립니다. label을 섞었을 때 exact-event EB의 gene×type EB 대비 이득이 `+0.01` 미만이면 PASS입니다. Specialist는 이 감사에서 제외합니다.

In [ ]:
# 오래 걸리는 선택 셀입니다. 수정된 함수를 반영하기 위해 모듈을 reload합니다.
import importlib
exp = importlib.reload(exp)
permutation_folds, permutation_report = exp.permutation_check(train, genes, seed=42)
display(permutation_folds)
permutation_report


In [ ]:
(RESULT_DIR / 'exact_event_permutation_report.json').write_text(
    __import__('json').dumps(permutation_report, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print('saved:', RESULT_DIR / 'exact_event_permutation_report.json')
